# Efficient Code Generation — Live Demo

**Columbia HPML Spring 2026** | Jasmine Truong · Yingxin Zhang · Arnav Mahajan · Jianyi Gao

This notebook demonstrates two results side-by-side:
1. **vLLM** is 21× faster than HuggingFace at inference time
2. Our fine-tuned model generates **correct code** that passes all tests

Run on **Colab A100**. Install cell can be run off-camera before recording.

In [ ]:
# Run this cell before recording — takes ~2 minutes
!pip install -q vllm transformers accelerate

In [ ]:
import gc
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

SYSTEM_PROMPT = (
    "Write a correct Python solution optimized for fast execution time. "
    "Use efficient algorithms and data structures to minimize runtime. "
    "Return only the code with no explanation."
)

# Problem: dataset_index=3551 from our test split
INSTRUCTION = 'def fibonacci(n):\n    """What is the algorithm to compute the nth Fibonacci number?"""'

TESTS = """assert fibonacci(0) == 0
assert fibonacci(1) == 1
assert fibonacci(2) == 1
assert fibonacci(3) == 2
assert fibonacci(4) == 3
assert fibonacci(5) == 5
assert fibonacci(6) == 8
assert fibonacci(7) == 13
assert fibonacci(8) == 21
assert fibonacci(9) == 34
assert fibonacci(10) == 55
assert fibonacci(11) == 89
assert fibonacci(12) == 144
assert fibonacci(13) == 233"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": INSTRUCTION},
]

print("Setup complete.")

## The Problem

```python
def fibonacci(n):
    """What is the algorithm to compute the nth Fibonacci number?"""
```

**14 test cases** — `fibonacci(0)` through `fibonacci(13)`.
We'll generate a solution and verify it passes all of them.

In [ ]:
# ── HuggingFace Inference ──────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", trust_remote_code=True)

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print("=" * 52)
print("  HuggingFace Transformers — generating...")
print("=" * 52)

t0 = time.perf_counter()
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        streamer=streamer,
    )
hf_latency = time.perf_counter() - t0

generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
hf_output = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"\n⏱  HuggingFace: {hf_latency:.2f}s")

## Now with vLLM...

In [ ]:
# ── Run this cell OFF-CAMERA (before recording) ────────────────────────────
del model
gc.collect()
torch.cuda.empty_cache()

from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_ID, gpu_memory_utilization=0.85)
sampling_params = SamplingParams(max_tokens=256, temperature=0.0)

# Warmup: compiles CUDA graphs so the timed call is instant
_ = llm.generate([prompt], sampling_params)
print("vLLM ready.")

In [ ]:
# ── ON-CAMERA: timed vLLM inference ────────────────────────────────────────
print("=" * 52)
print("  vLLM — generating...")
print("=" * 52)

t0 = time.perf_counter()
outputs = llm.generate([prompt], sampling_params)
vllm_latency = time.perf_counter() - t0

vllm_output = outputs[0].outputs[0].text
print(vllm_output)
print(f"\n⚡ vLLM: {vllm_latency:.3f}s  →  {hf_latency / vllm_latency:.0f}x faster than HuggingFace")

In [ ]:
# ── Run the Tests ──────────────────────────────────────────────────────────
print("=" * 52)
print("  Running 14 test cases on generated code...")
print("=" * 52)
print()

namespace = {}
try:
    exec(vllm_output, namespace)
    exec(TESTS, namespace)
    print("✓  All 14 tests passed")
except AssertionError as e:
    print(f"✗  Test failed: {e}")
except Exception as e:
    print(f"✗  Error: {type(e).__name__}: {e}")

In [ ]:
# ── Summary: GPU Utilization & Throughput ──────────────────────────────────
# Numbers from outputs/serving_full_results.csv (1,000 prompts, A100)
print()
print("━" * 48)
print(f"  {'Backend':<18} {'GPU Util':>9}  {'Memory':>9}")
print("━" * 48)
print(f"  {'HuggingFace':<18} {'37.6%':>9}  {'3.6 GB':>9}")
print(f"  {'vLLM':<18} {'97.2%':>9}  {'36.7 GB':>9}")
print("━" * 48)
print()
print(f"  Throughput:  HF   233 tok/s  →  vLLM  1,782 tok/s  ( 7.6×)")
print(f"  Latency:     HF   0.69s      →  vLLM  0.03s        (21× faster)")
print()